# 📥 Notebook 1: Data Schema & Ingestion
**Step 2 of the Customer Churn Prediction Pipeline**

This notebook loads the raw CSV, enforces strict data types, and saves a clean Parquet file for downstream use.

## 1.1 — Install / Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.3.3
numpy version: 2.3.5


## 1.2 — Define Schema
We strictly enforce dtypes to catch any data quality issues early and prevent silent type coercions.

In [2]:
SCHEMA = {
    "customer_id":            "string",
    "cycle_start":            "string",
    "cycle_end":              "string",
    "billing_amount":         "float64",
    "last_payment_days_ago":  "float64",
    "plan_tier":              "category",
    "tenure_months":          "float64",
    "monthly_usage_hours":    "float64",
    "active_days":            "float64",
    "login_count":            "float64",
    "avg_session_min":        "float64",
    "device_count":           "float64",
    "add_on_count":           "float64",
    "support_tickets":        "float64",
    "sla_breaches":           "float64",
    "promotions_redeemed":    "float64",
    "email_opens":            "float64",
    "email_clicks":           "float64",
    "last_campaign_days_ago": "float64",
    "nps_score":              "float64",
    "region":                 "category",
    "is_autopay":             "bool",
    "is_discounted":          "bool",
    "has_family_bundle":      "bool",
    "churned_next_cycle":     "int64",
}
print(f"Schema defined: {len(SCHEMA)} columns")

Schema defined: 25 columns


## 1.3 — Load CSV & Enforce Types

In [3]:
df = pd.read_csv("../data/churn_frame.csv")

# Convert booleans from string if needed
for col in ["is_autopay", "is_discounted", "has_family_bundle"]:
    if df[col].dtype == object:
        df[col] = df[col].map({"True": True, "False": False})

df = df.astype(SCHEMA)

print(f"Shape: {df.shape}")
print(f"\nChurn rate: {df['churned_next_cycle'].mean():.2%}")
print(f"\nDtypes:\n{df.dtypes}")

Shape: (8000, 25)

Churn rate: 12.05%

Dtypes:
customer_id               string[python]
cycle_start               string[python]
cycle_end                 string[python]
billing_amount                   float64
last_payment_days_ago            float64
plan_tier                       category
tenure_months                    float64
monthly_usage_hours              float64
active_days                      float64
login_count                      float64
avg_session_min                  float64
device_count                     float64
add_on_count                     float64
support_tickets                  float64
sla_breaches                     float64
promotions_redeemed              float64
email_opens                      float64
email_clicks                     float64
last_campaign_days_ago           float64
nps_score                        float64
region                          category
is_autopay                          bool
is_discounted                       bool
has_family

## 1.4 — Quick Sanity Checks

In [4]:
# Missing values
missing = df.isnull().sum()
print("=== Missing Values ===")
print(missing[missing > 0] if missing.any() else "✅ No missing values")

# Basic stats on target
print("\n=== Target Distribution ===")
print(df["churned_next_cycle"].value_counts())
print(f"\nChurn: {df['churned_next_cycle'].sum()} / {len(df)} = {df['churned_next_cycle'].mean():.2%}")

=== Missing Values ===
✅ No missing values

=== Target Distribution ===
churned_next_cycle
0    7036
1     964
Name: count, dtype: int64

Churn: 964 / 8000 = 12.05%


## 1.5 — Save as Parquet
Parquet is column-oriented, preserves dtypes, and loads ~10x faster than CSV.

In [5]:
import os
os.makedirs("../data", exist_ok=True)

df.to_parquet("../data/churn_frame.parquet", index=False)
print(f"✅ Saved → ../data/churn_frame.parquet")
print(f"   Parquet size: {os.path.getsize('../data/churn_frame.parquet') / 1024:.1f} KB")

# Reload and verify
df_check = pd.read_parquet("../data/churn_frame.parquet")
assert df_check.shape == df.shape, "Shape mismatch!"
print(f"✅ Verified round-trip: {df_check.shape}")

✅ Saved → ../data/churn_frame.parquet
   Parquet size: 278.5 KB
✅ Verified round-trip: (8000, 25)


## ✅ Summary
- Loaded **8,000 customer-cycle rows** with **25 columns**
- Enforced strict dtypes (category, float64, bool, int64)
- Saved clean Parquet → `data/churn_frame.parquet`
- Churn rate is **~12%** — a realistic imbalanced dataset

**Next:** `02_eda.ipynb` — Exploratory Data Analysis & Leakage Guardrails